In [2]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import soundfile as sf
from jiwer import wer
import os 
import sys
sys.path.append(os.path.abspath("../src"))
from utils import get_libri_file_list

import warnings
warnings.filterwarnings('ignore')

In [5]:
# load mô hình và processor từ Hugging Face
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h") # wraps a feature extractor and a tokenizer into a single processor.
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
DATASET_ROOT = "../data/LibriSpeech"
list = get_libri_file_list(DATASET_ROOT, ["dev-clean"])

In [8]:
FILE_INDEX = 0 
audio_path, org_transcript = list[FILE_INDEX]
audio_path, org_transcript 

('../data/LibriSpeech\\dev-clean\\1272\\128104\\1272-128104-0000.flac',
 'MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL')

In [9]:
# đọc file âm thanh (sample rate phải 16 kHz)
speech, rate = sf.read(audio_path)
assert rate == 16000

# xử lý đầu vào và dự đoán
inputs = processor(speech, sampling_rate=rate, return_tensors="pt", padding=True).input_values  
with torch.no_grad():
    logits = model(inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)
pred_transcript = processor.batch_decode(predicted_ids)

print("Transcription:", pred_transcript)
print("Original text:", org_transcript)

print(f"WER: {wer(org_transcript, pred_transcript)}")


Transcription: ['MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL']
Original text: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
WER: 0.0
